In [1]:
import boto3
from botocore.exceptions import ClientError, NoCredentialsError, ProfileNotFound

AWS_PROFILE = "training"
AWS_REGION = "us-east-1"

try:
    session = boto3.Session(
        profile_name=AWS_PROFILE,
        region_name=AWS_REGION
    )

    print("Region:", session.region_name)

    sts = session.client("sts")

    identity = sts.get_caller_identity()

    print("Account   :", identity["Account"])
    print("User ID   :", identity["UserId"])
    print("Principal :", identity["Arn"])

except ProfileNotFound as e:
    print("Profile problem:", e)

except NoCredentialsError:
    print("AWS credentials not found")

except ClientError as e:
    print("AWS error:", e)

Region: us-east-1
Account   : 608553548146
User ID   : AIDAY3ME6HFZKAX5ZDECC
Principal : arn:aws:iam::608553548146:user/cloud_user


In [2]:
s3 = session.client("s3")

try:
    response = s3.list_buckets()

    print("Available S3 buckets:")
    for bucket in response.get("Buckets", []):
        print(f"- {bucket['Name']}")

except NoCredentialsError:
    print("AWS credentials were not found.")
except ClientError as exc:
    print(f"AWS API error: {exc}")
except BotoCoreError as exc:
    print(f"AWS SDK error: {exc}")

Available S3 buckets:
- aws-glue-assets-608553548146-us-east-1
- gks-datalake2


In [ ]:
import json
import time
from botocore.exceptions import ClientError

# ---------------------------------------------------------
# Configuration
# ---------------------------------------------------------

KINESIS_STREAM_NAME = "gks-kinesis-stream"

POLL_INTERVAL = 1  # seconds


# ---------------------------------------------------------
# Kinesis Client
# ---------------------------------------------------------

kinesis_client = session.client("kinesis")


# ---------------------------------------------------------
# Find shards
# ---------------------------------------------------------

response = kinesis_client.describe_stream(
    StreamName=KINESIS_STREAM_NAME
)

shards = response["StreamDescription"]["Shards"]

print(f"Stream: {KINESIS_STREAM_NAME}")
print(f"Number of shards: {len(shards)}")


# ---------------------------------------------------------
# Create iterator for each shard
# ---------------------------------------------------------

shard_iterators = {}

for shard in shards:

    shard_id = shard["ShardId"]

    response = kinesis_client.get_shard_iterator(
        StreamName=KINESIS_STREAM_NAME,
        ShardId=shard_id,

        # Start with new records arriving after consumer starts
        ShardIteratorType="LATEST"
    )

    shard_iterators[shard_id] = response["ShardIterator"]

    print(f"Consumer attached to {shard_id}")


# ---------------------------------------------------------
# Consume continuously
# ---------------------------------------------------------

print("\nWaiting for records...\n")

while True:

    for shard_id, shard_iterator in list(shard_iterators.items()):

        try:

            response = kinesis_client.get_records(
                ShardIterator=shard_iterator,
                Limit=100
            )

            # Always update iterator
            shard_iterators[shard_id] = response["NextShardIterator"]

            records = response["Records"]

            for record in records:

                # Data is returned as bytes
                data = record["Data"].decode("utf-8")

                # Convert JSON string -> Python dictionary
                invoice = json.loads(data)

                print(
                    f"Shard={shard_id} | "
                    f"Sequence={record['SequenceNumber']}"
                )

                print(json.dumps(invoice, indent=2))
                print("-" * 60)

        except ClientError as e:
            print(f"Error reading {shard_id}: {e}")

    time.sleep(POLL_INTERVAL)

Stream: gks-kinesis-stream
Number of shards: 4
Consumer attached to shardId-000000000000
Consumer attached to shardId-000000000001
Consumer attached to shardId-000000000002
Consumer attached to shardId-000000000003

Waiting for records...

Shard=shardId-000000000003 | Sequence=49678545858277134489576005510010257058614805833149579314
{
  "InvoiceNo": "89488ABE",
  "StockCode": "85123A",
  "Quantity": 5,
  "Description": "TODO",
  "InvoiceDate": "2026-09-21T11:14:43.968393+00:00",
  "UnitPrice": 2.0,
  "CustomerID": 17850,
  "Country": "AT"
}
------------------------------------------------------------
Shard=shardId-000000000003 | Sequence=49678545858277134489576005510056196239760161741788414002
{
  "InvoiceNo": "89488ABE",
  "StockCode": "84406E",
  "Quantity": 1,
  "Description": "TODO",
  "InvoiceDate": "2026-09-21T11:14:43.968393+00:00",
  "UnitPrice": 4.0,
  "CustomerID": 17850,
  "Country": "AT"
}
------------------------------------------------------------
Shard=shardId-0000000000